In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from time import sleep

# Problem 2

In [82]:
def snow(M: int, prob: float , melt: bool, T: int, seed: int = None, N: int=None, arr: np.ndarray=None) -> list:
    
    def check_pos_int(x):
        if isinstance(x, int) and x > 0:
            return True
        else:
            return False
    
    assert check_pos_int(N) or N is None, "Wrong input for N"
    assert check_pos_int(M), "Wrong input for M"
    assert check_pos_int(seed) or seed is None, "Wrong input for seed"
    assert (isinstance(arr, np.ndarray) and isinstance(arr[0], int)) or arr is None, "Wrong input for arr"
    assert N or arr is not None, "At least one of the arguments: N, arr should be provided"
    assert (isinstance(prob, float) and 0 < prob < 1) or prob is None, "Wrong type of input for prob"
    assert isinstance(melt, bool), "Wrong type of input for melt variable"
    assert check_pos_int(T), "Wrong value for T"

    np.random.seed(seed)
    
    if N and arr is not None:
        assert N == arr.shape[0], "Wrong parameters, N != arr.shape[0]"
    elif N:
        arr = np.random.binomial(1, prob, N)
    else:
        N = arr.shape[0]

    space = np.zeros((M, N))

    vals = []
    
    for i in range(T):
        space[-1, :] += space[-2, :]
        space[1:-1,:] = space[:-2, :] 
        space[0, :] = arr
        arr = np.random.binomial(1, prob, N)
        # print(f'{space}')
        if melt and i % 3 == 0:
            space[-1, :][space[-1, :] > 0] -= 1
        vals.append(space.copy())
        
    
    return vals
        



In [86]:
vals = snow(N=2, M=3, prob=.5, melt=True, T=4, seed=42)
vals

[array([[0., 1.],
        [0., 0.],
        [0., 0.]]),
 array([[1., 1.],
        [0., 1.],
        [0., 0.]]),
 array([[0., 0.],
        [1., 1.],
        [0., 1.]]),
 array([[0., 1.],
        [0., 0.],
        [0., 1.]])]

In [76]:
from IPython.display import clear_output
clear_output(wait=True)


clear = lambda : system('cls')


for val in vals:
    sleep(.5)
    print(val)
    clear_output(wait=True)

[[1. 1.]
 [1. 1.]
 [2. 6.]]


# Problem 3

In [107]:
def fraud(views: pd.DataFrame, idx=None) -> pd.DataFrame:
    assert isinstance(views, pd.DataFrame), "Wrong type of input for views"
    vals = []
    for i in range(len(views)):
        vals.append(len(views[:i][views['author_id'] == views['viewer_id']][['author_id']]))
    plt.plot(vals)
    plt.xlabel('Row')
    plt.ylabel('Number of frauds')
    plt.title('Dependency of self-reviews on the number of lines taken')
    plt.savefig(f'example_{idx}.svg')
    plt.show()
    df = views[views['author_id'] == views['viewer_id']][['author_id']]
    df.rename(columns={'author_id' : 'id'}, inplace=True)
    df.drop_duplicates(inplace=True)
    df['id'] = sorted(df['id'])
    return df

# views = pd.DataFrame({'article_id' : [1, 1, 2, 2, 4, 3, 3],
#                       'author_id' : [3, 3, 7, 7, 7, 4, 4],
#                       'viewer_id' : [5, 6, 7, 6, 1, 4, 4], 
#                       'view_date' : ['2019-08-01', '2019-08-02', '2019-08-01', '2019-08-02', '2019-07-22', '2019-07-21', '2019-07-21']
#                      })
# res = pd.DataFrame({'id' : [4, 7]})

views_list, res_list = [], []

# Original example
views1 = pd.DataFrame({
    'article_id': [1, 1, 2, 2, 4, 3, 3],
    'author_id': [3, 3, 7, 7, 7, 4, 4],
    'viewer_id': [5, 6, 7, 6, 1, 4, 4], 
    'view_date': ['2019-08-01', '2019-08-02', '2019-08-01', '2019-08-02', 
                  '2019-07-22', '2019-07-21', '2019-07-21']
})
res1 = pd.DataFrame({'id': [4, 7]})

views_list.append(views1)
res_list.append(res1)

# Example 2: Multiple fraud authors
views2 = pd.DataFrame({
    'article_id': [1, 1, 2, 2, 3, 3, 4, 4, 5, 5],
    'author_id': [3, 3, 7, 7, 4, 4, 2, 2, 9, 9],
    'viewer_id': [5, 3, 7, 7, 4, 4, 2, 8, 9, 9],
    'view_date': ['2019-08-01', '2019-08-01', '2019-08-02', '2019-08-02', 
                  '2019-08-03', '2019-08-03', '2019-08-04', '2019-08-04',
                  '2019-08-05', '2019-08-05']
})
res2 = pd.DataFrame({'id': [2, 3, 4, 7, 9]})

views_list.append(views2)
res_list.append(res2)


# Example 3: No fraud cases
views3 = pd.DataFrame({
    'article_id': [1, 2, 3, 4, 5],
    'author_id': [1, 2, 3, 4, 5],
    'viewer_id': [6, 7, 8, 9, 10],
    'view_date': ['2019-08-01', '2019-08-02', '2019-08-03', 
                  '2019-08-04', '2019-08-05']
})
res3 = pd.DataFrame({'id': []})

views_list.append(views3)
res_list.append(res3)

# Example 4: Single author with multiple self-reviews
views4 = pd.DataFrame({
    'article_id': [1, 2, 3, 4, 5, 6],
    'author_id': [5, 5, 5, 5, 5, 5],
    'viewer_id': [5, 1, 2, 5, 3, 5],
    'view_date': ['2019-08-01', '2019-08-02', '2019-08-03',
                  '2019-08-04', '2019-08-05', '2019-08-06']
})
res4 = pd.DataFrame({'id': [5]})

views_list.append(views4)
res_list.append(res4)

# Example 5: Mixed valid and invalid views
views5 = pd.DataFrame({
    'article_id': [1, 2, 3, 4, 5, 6, 7, 8],
    'author_id': [10, 11, 12, 13, 10, 11, 12, 13],
    'viewer_id': [10, 20, 12, 30, 40, 11, 50, 13],
    'view_date': ['2019-08-01', '2019-08-01', '2019-08-01', '2019-08-01',
                  '2019-08-02', '2019-08-02', '2019-08-02', '2019-08-02']
})
res5 = pd.DataFrame({'id': [10, 11, 12, 13]})

views_list.append(views5)
res_list.append(res5)

# Example 6: Large dataset with pattern
views6 = pd.DataFrame({
    'article_id': list(range(1, 21)),
    'author_id': [i % 5 + 1 for i in range(20)],
    'viewer_id': [(i % 5 + 1) if i % 3 == 0 else (i % 10 + 11) for i in range(20)],
    'view_date': [f'2019-08-{str(i%30+1).zfill(2)}' for i in range(20)]
})
# Expected frauds: authors who viewed their own articles (when i % 3 == 0)
# Authors: 1, 2, 3, 4, 5
# Self-views happen when author_id = viewer_id
res6 = pd.DataFrame({'id': [1, 2, 3, 4, 5]})

views_list.append(views6)
res_list.append(res6)

from IPython.display import clear_output
from os import system
clear_output(wait=True)

clear = lambda : system('cls')

for idx, (views, res) in enumerate(zip(views_list, res_list)):
    assert np.allclose(fraud(views, idx+1), res), "!!!"
    clear_output(wait=True)

print('All good!')

all good!


# Problem 4

In [191]:
class int4():
    
    def __init__(self, num: int = None) -> None:
        if num is None:
            self.num = [0]
        else:
            self.num = self.to_base4(num)

    @staticmethod
    def to_base4(num: int) -> list:
        b4 = []
        while num >= 4:
            b4.append(num % 4)
            num //= 4
        b4.append(num)
        return b4[::-1]

    @staticmethod
    def to_base10(b4: list) -> int:
        res = 0
        for idx, num in enumerate(b4[::-1]):
            res += 4**idx * num
        return res

    def __add__(self, value) -> list:
        # assert isinstance(value, int4), "!"
        res = []
        sm, rem = 0, 0
        max_len = max(len(self.num), len(value.num))
        if len(self.num) == max_len:
            pad = [0]*(max_len - len(value.num))
            value.num = pad + value.num
            value.num = value.num.copy()
        else:
            pad = [0]*(max_len - len(self.num))
            self.num = pad + self.num
            self.num = self.num.copy()
        for i, j in zip(self.num[::-1], value.num[::-1]):
            if i is not None and j is not None:
                sm = (i + j) + rem
            elif i is None:
                sm = j + rem
            else:
                sm = i + rem
            # print(sm, rem, res)
            
            rem = sm // 4
            sm %= 4
            res.append(sm)
            
        if rem != 0:
            
            res.append(rem)
            
        # print(res)
        res = self.to_base10(res[::-1])
        
        return int4(res)

    def __repr__(self):
        val = []
        for i in self.num:
            val.append(str(i))
        
        return ''.join(val)

    def replace(self, num: int) -> None:
        self.__init__(num)

    def __eq__(self, value) -> None:
        for i, j in zip(self.num, value.num):
            if i != j:
                return False
        return True
            

a = int4(10)
b = int4(15)
c = a+b
print(a, b, c)

22 33 121


In [193]:
assert int4().num == [0]

assert int4(0).num == [0]
assert int4(1).num == [1]
assert int4(2).num == [2]
assert int4(3).num == [3]
assert int4(4).num == [1, 0]
assert int4(5).num == [1, 1]
assert int4(6).num == [1, 2]
assert int4(7).num == [1, 3]
assert int4(8).num == [2, 0]
assert int4(9).num == [2, 1]
assert int4(10).num == [2, 2]
assert int4(11).num == [2, 3]
assert int4(12).num == [3, 0]
assert int4(13).num == [3, 1]
assert int4(14).num == [3, 2]
assert int4(15).num == [3, 3]

a = int4(1)   # [1]
b = int4(4)   # [1,0]
c = a + b     # Should be [1,1] = 5
assert c.num == [1, 1]

a = int4(4)   # [1,0]
b = int4(1)   # [1]
c = a + b     # Should be [1,1] = 5
assert c.num == [1, 1]

# Test 3: 16 + 1 = 17
# 16 = [1,0,0], 1 = [1] -> padded: [0,0,1] + [1,0,0] = [1,0,1] = 17
a = int4(16)  # [1,0,0]
b = int4(1)   # [1]
c = a + b     # Should be [1,0,1] = 17
assert c.num == [1, 0, 1]

assert (int4(0) + int4(0)).num == [0]
assert (int4(1) + int4(2)).num == [3]
assert (int4(2) + int4(2)).num == [1, 0]  # 4 in base-4
assert (int4(3) + int4(1)).num == [1, 0]  # 4 in base-4
assert (int4(3) + int4(2)).num == [1, 1]  # 5 in base-4
assert (int4(3) + int4(3)).num == [1, 2]  # 6 in base-4

# 4 + 4 = 8 = [2,0]
assert (int4(4) + int4(4)).num == [2, 0]
# 5 + 5 = 10 = [2,2]
assert (int4(5) + int4(5)).num == [2, 2]
# 7 + 5 = 12 = [3,0]
assert (int4(7) + int4(5)).num == [3, 0]
# 7 + 7 = 14 = [3,2]
assert (int4(7) + int4(7)).num == [3, 2]

# 16 + 16 = 32 = [2,0,0]
assert (int4(16) + int4(16)).num == [2, 0, 0]
# 20 + 10 = 30 = [1,3,2]
assert (int4(20) + int4(10)).num == [1, 3, 2]
# 25 + 25 = 50 = [3,0,2]
assert (int4(25) + int4(25)).num == [3, 0, 2]
# 31 + 31 = 62 = [3,3,2]
assert (int4(31) + int4(31)).num == [3, 3, 2]

# 3 + 1 = 4 (carry from units to fours)
assert (int4(3) + int4(1)).num == [1, 0]

# 7 + 1 = 8 (7=13 base4, +1 = 20 base4)
assert (int4(7) + int4(1)).num == [2, 0]

# 11 + 1 = 12 (11=23 base4, +1 = 30 base4)
assert (int4(11) + int4(1)).num == [3, 0]

# 3 + 3 = 6 (carry 1, result 12 base4)
assert (int4(3) + int4(3)).num == [1, 2]

# 7 + 5 = 12 (7=13 base4, 5=11 base4, sum=30 base4)
assert (int4(7) + int4(5)).num == [3, 0]

# 15 + 9 = 24 (15=33 base4, 9=21 base4, sum=120 base4)
assert (int4(15) + int4(9)).num == [1, 2, 0]

# 15 + 1 = 16 (carry creates new digit)
assert (int4(15) + int4(1)).num == [1, 0, 0]

# 31 + 1 = 32 (carry creates new digit)
assert (int4(31) + int4(1)).num == [2, 0, 0]

# 63 + 1 = 64 (carry creates new digit)
assert (int4(63) + int4(1)).num == [1, 0, 0, 0]

assert (int4(0) + int4(0)).num == [0]
assert (int4(5) + int4(0)).num == [1, 1]
assert (int4(0) + int4(5)).num == [1, 1]
assert (int4(16) + int4(0)).num == [1, 0, 0]
assert (int4(0) + int4(16)).num == [1, 0, 0]

assert (int4(3) + int4(3)).num == [1, 2]  # Max digit + Max digit

assert (int4(3) + int4(1)).num == [1, 0]      # 4 = 4^1
assert (int4(15) + int4(1)).num == [1, 0, 0]  # 16 = 4^2
assert (int4(63) + int4(1)).num == [1, 0, 0, 0]  # 64 = 4^3

assert (int4(1) + int4(2)).num == (int4(2) + int4(1)).num
assert (int4(5) + int4(3)).num == (int4(3) + int4(5)).num
assert (int4(10) + int4(15)).num == (int4(15) + int4(10)).num
assert (int4(20) + int4(25)).num == (int4(25) + int4(20)).num
assert (int4(16) + int4(32)).num == (int4(32) + int4(16)).num

test_numbers = [0, 1, 3, 4, 5, 10, 16, 25, 31, 63]
for num in test_numbers:
    a = int4(num)
    zero = int4(0)
    assert (a + zero).num == a.num
    assert (zero + a).num == a.num

a = int4(10)  # [2,2]
b = int4(15)  # [3,3]
c = a + b     # Should be [1,2,1] = 25
assert c.num == [1, 2, 1]
assert int4.to_base10(c.num) == 25

a = int4(27)  # [1,2,3]
b = int4(30)  # [1,3,2]
c = a + b     # Should be [3,2,1] = 57
assert c.num == [3, 2, 1]
assert int4.to_base10(c.num) == 57

for a in range(16):
    for b in range(16):
        int4_a = int4(a)
        int4_b = int4(b)
        int4_sum = int4_a + int4_b
        decimal_sum = a + b
        
        # Verify using to_base10
        calculated_sum = int4.to_base10(int4_sum.num)
        assert calculated_sum == decimal_sum, f"{a} + {b} = {decimal_sum}, but got {calculated_sum}"
        
        # Verify using constructor
        expected_int4 = int4(decimal_sum)
        assert int4_sum.num == expected_int4.num, f"{a} + {b}: expected {expected_int4.num}, got {int4_sum.num}"

# Test all two-digit base-4 numbers (4-15)
for a in range(4, 16):
    for b in range(4, 16):
        int4_a = int4(a)
        int4_b = int4(b)
        int4_sum = int4_a + int4_b
        decimal_sum = a + b
        
        expected_int4 = int4(decimal_sum)
        assert int4_sum.num == expected_int4.num

# Test all three-digit base-4 numbers (16-63)
for a in range(16, 64, 7):  # Step to reduce test count
    for b in range(16, 64, 7):
        int4_a = int4(a)
        int4_b = int4(b)
        int4_sum = int4_a + int4_b
        decimal_sum = a + b
        
        expected_int4 = int4(decimal_sum)
        assert int4_sum.num == expected_int4.num

# 3 + 1 = 4 (crosses from 1 to 2 digits)
assert (int4(3) + int4(1)).num == [1, 0]

# 15 + 1 = 16 (crosses from 2 to 3 digits)
assert (int4(15) + int4(1)).num == [1, 0, 0]

# 63 + 1 = 64 (crosses from 3 to 4 digits)
assert (int4(63) + int4(1)).num == [1, 0, 0, 0]

# Max 1-digit: 3 + 3 = 6
assert (int4(3) + int4(3)).num == [1, 2]

# Max 2-digit: 15 + 15 = 30
assert (int4(15) + int4(15)).num == [1, 3, 2]

# Max 3-digit: 63 + 63 = 126
assert (int4(63) + int4(63)).num == [1, 3, 3, 2]

assert repr(int4(0)) == "0"
assert repr(int4(5)) == "11"
assert repr(int4(10)) == "22"
assert repr(int4(16)) == "100"
assert repr(int4(25)) == "121"
assert repr(int4(57)) == "321"

assert int4(0) == int4(0)
assert int4(5) == int4(5)
assert int4(10) == int4(10)
assert int4(25) == int4(25)

assert not (int4(0) == int4(1))
assert not (int4(5) == int4(6))
assert not (int4(10) == int4(11))

a = int4(5)
assert a.num == [1, 1]
a.replace(10)
assert a.num == [2, 2]
a.replace(0)
assert a.num == [0]
a.replace(16)
assert a.num == [1, 0, 0]

# Test that replace doesn't affect other operations
a = int4(5)
b = int4(3)
c = a + b  # 5 + 3 = 8
assert c.num == [2, 0]
a.replace(10)
d = a + b  # Now 10 + 3 = 13
assert d.num == [3, 1]

# Create numbers
a = int4(5)    # 11 base4
b = int4(3)    # 3 base4
c = int4(12)   # 30 base4

# Test chain of operations
result1 = a + b      # 5 + 3 = 8 = 20 base4
assert result1.num == [2, 0]

result2 = result1 + c  # 8 + 12 = 20 = 110 base4
assert result2.num == [1, 1, 0]

# Test with replacement
a.replace(7)         # Now a is 7 = 13 base4
result3 = a + b      # 7 + 3 = 10 = 22 base4
assert result3.num == [2, 2]

# Test equality after operations
assert result3 == int4(10)

print("ALL TESTS PASSED!")

ALL TESTS PASSED!


In [2]:
# Reference implementation for t_anal function
def t_anal(data, scal):
    """
    Reference implementation of t_anal function for testing.
    """
    # Step 1: Handle NaN values
    cleaned = data.copy()
    for i in range(cleaned.shape[0]):
        row = cleaned[i]
        non_nan_mask = ~np.isnan(row)
        if np.any(non_nan_mask):
            row_mean = np.nanmean(row)
            row[np.isnan(row)] = row_mean
        else:
            # Entire row is NaN
            row[:] = 0
        cleaned[i] = row
    
    # Step 2: Apply transformation
    def transform_row(row):
        # Convert negatives to absolute values
        row = np.abs(row)
        # Round to 2 decimal places
        row = np.round(row, 2)
        # Multiply by scalar
        row = row * scal 
        return row
    
    transformed = np.apply_along_axis(transform_row, axis=1, arr=cleaned)
    
    # Step 3a: Add weekly total column
    weekly_totals = np.sum(transformed, axis=1, keepdims=True)
    with_totals = np.hstack([transformed, weekly_totals])
    
    # Step 3b: Add daily averages row
    daily_avgs = np.mean(with_totals, axis=0, keepdims=True)
    augmented = np.vstack([with_totals, daily_avgs])
    
    # Step 3c: Rearrange columns (weekends first)
    weekend_data = transformed[:, 5:7]  # Last 2 columns
    weekday_data = transformed[:, 0:5]  # First 5 columns
    rearranged = np.hstack([weekend_data, weekday_data])
    
    return transformed, augmented, rearranged

In [3]:
# Test Case 1: Basic case with some NaNs and positive values
data1 = np.array([
    [10.123, 12.345, np.nan, 14.567, 11.890, 15.001, 13.789],
    [8.765, 9.210, 10.500, 10.123, np.nan, 11.567, 9.876],
    [20.0, 22.222, 19.999, 21.500, 20.900, np.nan, 23.333],
    [5.555, 6.666, 7.777, 5.888, 6.999, 7.500, np.nan]
], dtype=np.float64)
scal1 = 1.5
expected1 = (
    np.array([
        [15.18, 18.52, 13.30, 21.85, 17.84, 22.50, 20.68],
        [13.15, 13.82, 15.75, 15.18, 11.39, 17.35, 14.81],
        [30.00, 33.33, 30.00, 32.25, 31.35, 26.37, 35.00],
        [8.33, 10.00, 11.67, 8.83, 10.50, 11.25, 9.44]
    ]),
    np.array([
        [15.18, 18.52, 13.30, 21.85, 17.84, 22.50, 20.68, 129.88],
        [13.15, 13.82, 15.75, 15.18, 11.39, 17.35, 14.81, 101.46],
        [30.00, 33.33, 30.00, 32.25, 31.35, 26.37, 35.00, 218.30],
        [8.33, 10.00, 11.67, 8.83, 10.50, 11.25, 9.44, 70.02],
        [16.67, 18.92, 17.68, 19.53, 17.77, 19.37, 19.98, 129.92]
    ]),
    np.array([
        [22.50, 20.68, 15.18, 18.52, 13.30, 21.85, 17.84],
        [17.35, 14.81, 13.15, 13.82, 15.75, 15.18, 11.39],
        [26.37, 35.00, 30.00, 33.33, 30.00, 32.25, 31.35],
        [11.25, 9.44, 8.33, 10.00, 11.67, 8.83, 10.50]
    ])
)

# Test Case 2: Contains negative values
data2 = np.array([
    [10.0, -5.5, 15.25, -20.75, 25.0, np.nan, 30.5],
    [-1.5, -2.75, -3.25, np.nan, -4.5, -5.75, -6.0],
    [100.0, 200.0, -300.5, 400.25, np.nan, -500.0, 600.75],
    [0.0, -0.75, 0.50, np.nan, -1.25, 1.75, np.nan]
], dtype=np.float64)
scal2 = 2.0
expected2 = (
    np.array([
        [20.00, 11.00, 30.50, 41.50, 50.00, 28.65, 61.00],
        [3.00, 5.50, 6.50, 4.71, 9.00, 11.50, 12.00],
        [200.00, 400.00, 601.00, 800.50, 450.28, 1000.00, 1201.50],
        [0.00, 1.50, 1.00, 0.50, 2.50, 3.50, 1.67]
    ]),
    np.array([
        [20.00, 11.00, 30.50, 41.50, 50.00, 28.65, 61.00, 242.65],
        [3.00, 5.50, 6.50, 4.71, 9.00, 11.50, 12.00, 52.21],
        [200.00, 400.00, 601.00, 800.50, 450.28, 1000.00, 1201.50, 4653.28],
        [0.00, 1.50, 1.00, 0.50, 2.50, 3.50, 1.67, 10.67],
        [55.75, 104.50, 159.75, 211.80, 127.95, 260.91, 319.04, 1243.20]
    ]),
    np.array([
        [28.65, 61.00, 20.00, 11.00, 30.50, 41.50, 50.00],
        [11.50, 12.00, 3.00, 5.50, 6.50, 4.71, 9.00],
        [1000.00, 1201.50, 200.00, 400.00, 601.00, 800.50, 450.28],
        [3.50, 1.67, 0.00, 1.50, 1.00, 0.50, 2.50]
    ])
)

# Test Case 3: Many NaNs in the data
data3 = np.array([
    [np.nan, np.nan, 5.123, np.nan, np.nan, 10.456, np.nan],
    [1.111, 2.222, np.nan, np.nan, np.nan, np.nan, 3.333],
    [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 100.0],
    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
], dtype=np.float64)
scal3 = 0.5
expected3 = (
    np.array([
        [1.28, 1.28, 2.56, 1.28, 1.28, 2.56, 1.28],
        [0.56, 1.11, 1.11, 1.11, 1.11, 1.11, 1.67],
        [50.00, 50.00, 50.00, 50.00, 50.00, 50.00, 50.00],
        [0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00]
    ]),
    np.array([
        [1.28, 1.28, 2.56, 1.28, 1.28, 2.56, 1.28, 11.52],
        [0.56, 1.11, 1.11, 1.11, 1.11, 1.11, 1.67, 7.78],
        [50.00, 50.00, 50.00, 50.00, 50.00, 50.00, 50.00, 350.00],
        [0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00],
        [12.96, 13.10, 13.42, 13.10, 13.10, 13.42, 13.24, 92.33]
    ]),
    np.array([
        [2.56, 1.28, 1.28, 1.28, 2.56, 1.28, 1.28],
        [1.11, 1.67, 0.56, 1.11, 1.11, 1.11, 1.11],
        [50.00, 50.00, 50.00, 50.00, 50.00, 50.00, 50.00],
        [0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00]
    ])
)

# Test Case 4: Entire row is NaN
data4 = np.array([
    [10.1, 20.2, 30.3, 40.4, 50.5, 60.6, 70.7],
    [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan],
    [1.11, 2.22, 3.33, 4.44, 5.55, 6.66, 7.77],
    [15.0, 25.0, 35.0, 45.0, 55.0, 65.0, 75.0]
], dtype=np.float64)
scal4 = 1.0
expected4 = (
    np.array([
        [10.10, 20.20, 30.30, 40.40, 50.50, 60.60, 70.70],
        [0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00],
        [1.11, 2.22, 3.33, 4.44, 5.55, 6.66, 7.77],
        [15.00, 25.00, 35.00, 45.00, 55.00, 65.00, 75.00]
    ]),
    np.array([
        [10.10, 20.20, 30.30, 40.40, 50.50, 60.60, 70.70, 282.80],
        [0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00],
        [1.11, 2.22, 3.33, 4.44, 5.55, 6.66, 7.77, 31.08],
        [15.00, 25.00, 35.00, 45.00, 55.00, 65.00, 75.00, 315.00],
        [6.55, 11.86, 17.16, 22.46, 27.76, 33.07, 38.37, 157.22]
    ]),
    np.array([
        [60.60, 70.70, 10.10, 20.20, 30.30, 40.40, 50.50],
        [0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00],
        [6.66, 7.77, 1.11, 2.22, 3.33, 4.44, 5.55],
        [65.00, 75.00, 15.00, 25.00, 35.00, 45.00, 55.00]
    ])
)

# Test Case 5: Decimal precision and rounding test
data5 = np.array([
    [1.234567, 2.345678, 3.456789, np.nan, 4.567890, 5.678901, 6.789012],
    [0.00123, 0.00234, 0.00345, 0.00456, 0.00567, np.nan, 0.00678],
    [999.999, -888.888, 777.777, np.nan, -666.666, 555.555, 444.444],
    [np.nan, np.nan, np.nan, 1.111, 2.222, 3.333, 4.444]
], dtype=np.float64)
scal5 = 1.25
expected5 = (
    np.array([
        [2.02, 3.54, 5.41, 3.92, 7.15, 8.90, 10.61],
        [0.00, 0.00, 0.00, 0.01, 0.01, 0.00, 0.01],
        [1249.99, 1111.11, 972.22, 833.34, 833.33, 694.44, 555.56],
        [1.39, 1.39, 1.39, 1.39, 2.78, 4.17, 5.56]
    ]),
    np.array([
        [2.02, 3.54, 5.41, 3.92, 7.15, 8.90, 10.61, 41.55],
        [0.00, 0.00, 0.00, 0.01, 0.01, 0.00, 0.01, 0.04],
        [1249.99, 1111.11, 972.22, 833.34, 833.33, 694.44, 555.56, 6250.00],
        [1.39, 1.39, 1.39, 1.39, 2.78, 4.17, 5.56, 18.07],
        [313.35, 279.26, 244.76, 209.67, 210.82, 176.88, 142.94, 1577.67]
    ]),
    np.array([
        [8.90, 10.61, 2.02, 3.54, 5.41, 3.92, 7.15],
        [0.00, 0.01, 0.00, 0.00, 0.00, 0.01, 0.01],
        [694.44, 555.56, 1249.99, 1111.11, 972.22, 833.34, 833.33],
        [4.17, 5.56, 1.39, 1.39, 1.39, 1.39, 2.78]
    ])
)

# Test Case 6: Mixed values including zeros
data6 = np.array([
    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0],
    [-1.0, -2.0, -3.0, -4.0, -5.0, -6.0, -7.0],
    [np.nan, 10.0, np.nan, 20.0, np.nan, 30.0, np.nan]
], dtype=np.float64)
scal6 = 3.0
expected6 = (
    np.array([
        [0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00],
        [3.00, 6.00, 9.00, 12.00, 15.00, 18.00, 21.00],
        [3.00, 6.00, 9.00, 12.00, 15.00, 18.00, 21.00],
        [15.00, 30.00, 15.00, 60.00, 15.00, 90.00, 15.00]
    ]),
    np.array([
        [0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00],
        [3.00, 6.00, 9.00, 12.00, 15.00, 18.00, 21.00, 84.00],
        [3.00, 6.00, 9.00, 12.00, 15.00, 18.00, 21.00, 84.00],
        [15.00, 30.00, 15.00, 60.00, 15.00, 90.00, 15.00, 240.00],
        [5.25, 10.50, 8.25, 21.00, 11.25, 31.50, 14.25, 102.00]
    ]),
    np.array([
        [0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00],
        [18.00, 21.00, 3.00, 6.00, 9.00, 12.00, 15.00],
        [18.00, 21.00, 3.00, 6.00, 9.00, 12.00, 15.00],
        [90.00, 15.00, 15.00, 30.00, 15.00, 60.00, 15.00]
    ])
)

# Test Case 7: Small scalar value
data7 = np.array([
    [100.0, 200.0, 300.0, 400.0, 500.0, 600.0, 700.0],
    [10.0, 20.0, 30.0, 40.0, 50.0, 60.0, 70.0],
    [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0],
    [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
], dtype=np.float64)
scal7 = 0.1
expected7 = (
    np.array([
        [10.00, 20.00, 30.00, 40.00, 50.00, 60.00, 70.00],
        [1.00, 2.00, 3.00, 4.00, 5.00, 6.00, 7.00],
        [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70],
        [0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07]
    ]),
    np.array([
        [10.00, 20.00, 30.00, 40.00, 50.00, 60.00, 70.00, 280.00],
        [1.00, 2.00, 3.00, 4.00, 5.00, 6.00, 7.00, 28.00],
        [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 2.80],
        [0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.28],
        [2.78, 5.56, 8.33, 11.11, 13.89, 16.67, 19.44, 77.78]
    ]),
    np.array([
        [60.00, 70.00, 10.00, 20.00, 30.00, 40.00, 50.00],
        [6.00, 7.00, 1.00, 2.00, 3.00, 4.00, 5.00],
        [0.60, 0.70, 0.10, 0.20, 0.30, 0.40, 0.50],
        [0.06, 0.07, 0.01, 0.02, 0.03, 0.04, 0.05]
    ])
)

# Test Case 8: Large scalar value
data8 = np.array([
    [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0],
    [10.0, 20.0, 30.0, 40.0, 50.0, 60.0, 70.0],
    [0.5, 1.5, 2.5, 3.5, 4.5, 5.5, 6.5],
    [np.nan, 15.0, np.nan, 25.0, np.nan, 35.0, np.nan]
], dtype=np.float64)
scal8 = 10.0
expected8 = (
    np.array([
        [10.00, 20.00, 30.00, 40.00, 50.00, 60.00, 70.00],
        [100.00, 200.00, 300.00, 400.00, 500.00, 600.00, 700.00],
        [5.00, 15.00, 25.00, 35.00, 45.00, 55.00, 65.00],
        [25.00, 150.00, 25.00, 250.00, 25.00, 350.00, 25.00]
    ]),
    np.array([
        [10.00, 20.00, 30.00, 40.00, 50.00, 60.00, 70.00, 280.00],
        [100.00, 200.00, 300.00, 400.00, 500.00, 600.00, 700.00, 2800.00],
        [5.00, 15.00, 25.00, 35.00, 45.00, 55.00, 65.00, 245.00],
        [25.00, 150.00, 25.00, 250.00, 25.00, 350.00, 25.00, 850.00],
        [35.00, 96.25, 95.00, 181.25, 155.00, 266.25, 215.00, 1043.75]
    ]),
    np.array([
        [60.00, 70.00, 10.00, 20.00, 30.00, 40.00, 50.00],
        [600.00, 700.00, 100.00, 200.00, 300.00, 400.00, 500.00],
        [55.00, 65.00, 5.00, 15.00, 25.00, 35.00, 45.00],
        [350.00, 25.00, 25.00, 150.00, 25.00, 250.00, 25.00]
    ])
)

# Test Case 9: All values negative
data9 = np.array([
    [-1.0, -2.0, -3.0, -4.0, -5.0, -6.0, -7.0],
    [-10.0, -20.0, -30.0, -40.0, -50.0, -60.0, -70.0],
    [-0.1, -0.2, -0.3, -0.4, -0.5, -0.6, -0.7],
    [np.nan, -5.5, np.nan, -15.5, np.nan, -25.5, np.nan]
], dtype=np.float64)
scal9 = 1.0
expected9 = (
    np.array([
        [1.00, 2.00, 3.00, 4.00, 5.00, 6.00, 7.00],
        [10.00, 20.00, 30.00, 40.00, 50.00, 60.00, 70.00],
        [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70],
        [15.50, 5.50, 15.50, 15.50, 15.50, 25.50, 15.50]
    ]),
    np.array([
        [1.00, 2.00, 3.00, 4.00, 5.00, 6.00, 7.00, 28.00],
        [10.00, 20.00, 30.00, 40.00, 50.00, 60.00, 70.00, 280.00],
        [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 2.80],
        [15.50, 5.50, 15.50, 15.50, 15.50, 25.50, 15.50, 108.50],
        [6.65, 7.00, 12.20, 15.98, 18.75, 23.03, 23.30, 106.90]
    ]),
    np.array([
        [6.00, 7.00, 1.00, 2.00, 3.00, 4.00, 5.00],
        [60.00, 70.00, 10.00, 20.00, 30.00, 40.00, 50.00],
        [0.60, 0.70, 0.10, 0.20, 0.30, 0.40, 0.50],
        [25.50, 15.50, 15.50, 5.50, 15.50, 15.50, 15.50]
    ])
)

# Test Case 10: Random data with various patterns
data10 = np.array([
    [12.34, 23.45, 34.56, 45.67, 56.78, 67.89, 78.90],
    [98.76, 87.65, 76.54, 65.43, 54.32, 43.21, 32.10],
    [-10.1, 20.2, -30.3, 40.4, -50.5, 60.6, -70.7],
    [np.nan, 5.0, np.nan, 15.0, np.nan, 25.0, np.nan]
], dtype=np.float64)
scal10 = 1.75
expected10 = (
    np.array([
        [21.60, 41.04, 60.48, 79.92, 99.37, 118.81, 138.08],
        [172.83, 153.39, 133.95, 114.50, 95.06, 75.62, 56.17],
        [17.68, 35.35, 53.03, 70.70, 88.38, 106.05, 123.73],
        [11.25, 8.75, 11.25, 26.25, 11.25, 43.75, 11.25]
    ]),
    np.array([
        [21.60, 41.04, 60.48, 79.92, 99.37, 118.81, 138.08, 559.30],
        [172.83, 153.39, 133.95, 114.50, 95.06, 75.62, 56.17, 801.52],
        [17.68, 35.35, 53.03, 70.70, 88.38, 106.05, 123.73, 494.92],
        [11.25, 8.75, 11.25, 26.25, 11.25, 43.75, 11.25, 123.75],
        [55.84, 59.63, 64.68, 72.84, 73.52, 86.06, 82.31, 494.87]
    ]),
    np.array([
        [118.81, 138.08, 21.60, 41.04, 60.48, 79.92, 99.37],
        [75.62, 56.17, 172.83, 153.39, 133.95, 114.50, 95.06],
        [106.05, 123.73, 17.68, 35.35, 53.03, 70.70, 88.38],
        [43.75, 11.25, 11.25, 8.75, 11.25, 26.25, 11.25]
    ])
)

test_cases = [
    ("Test Case 1: Basic case", data1, scal1, expected1),
    ("Test Case 2: Contains negative values", data2, scal2, expected2),
    ("Test Case 3: Many NaNs", data3, scal3, expected3),
    ("Test Case 4: Entire row is NaN", data4, scal4, expected4),
    ("Test Case 5: Decimal precision", data5, scal5, expected5),
    ("Test Case 6: Mixed values including zeros", data6, scal6, expected6),
    ("Test Case 7: Small scalar value", data7, scal7, expected7),
    ("Test Case 8: Large scalar value", data8, scal8, expected8),
    ("Test Case 9: All values negative", data9, scal9, expected9),
    ("Test Case 10: Random data with patterns", data10, scal10, expected10)
            ]
    
print("Testing implementation...")

for i, (description, data, scal, expected) in enumerate(test_cases):
    
    result = t_anal(data, scal)
    
    for arr_name, result_arr, expected_arr in zip(
        ["Transformed", "Augmented", "Rearranged"],
        result,
        expected
    ):
        # if not np.allclose(result_arr, expected_arr, rtol=1e-2):
        #     print(f'You got an error in test {i}!')
        #     break

        print(expected_arr, '\n\n', result_arr)
        print('\n\n\n\n\n------------------------------')

print("All test cases completed!")

Testing implementation...
[[15.18 18.52 13.3  21.85 17.84 22.5  20.68]
 [13.15 13.82 15.75 15.18 11.39 17.35 14.81]
 [30.   33.33 30.   32.25 31.35 26.37 35.  ]
 [ 8.33 10.   11.67  8.83 10.5  11.25  9.44]] 

 [[15.18  18.51  19.425 21.855 17.835 22.5   20.685]
 [13.14  13.815 15.75  15.18  15.015 17.355 14.82 ]
 [30.    33.33  30.    32.25  31.35  31.995 34.995]
 [ 8.34  10.005 11.67   8.835 10.5   11.25  10.095]]





------------------------------
[[ 15.18  18.52  13.3   21.85  17.84  22.5   20.68 129.88]
 [ 13.15  13.82  15.75  15.18  11.39  17.35  14.81 101.46]
 [ 30.    33.33  30.    32.25  31.35  26.37  35.   218.3 ]
 [  8.33  10.    11.67   8.83  10.5   11.25   9.44  70.02]
 [ 16.67  18.92  17.68  19.53  17.77  19.37  19.98 129.92]] 

 [[ 15.18     18.51     19.425    21.855    17.835    22.5      20.685
  135.99   ]
 [ 13.14     13.815    15.75     15.18     15.015    17.355    14.82
  105.075  ]
 [ 30.       33.33     30.       32.25     31.35     31.995    34.995
  223.92   

In [ ]:
def top_travellers(users: pd.DataFrame, rides: pd.DataFrame) -> pd.DataFrame:
    df = users.merge(rides, how='left', left_on='id', right_on='user_id')
    df = df[['name', 'id_x', 'distance']].groupby(['name', 'id_x']).sum().reset_index()
    df.drop(columns='id_x', inplace=True)
    df = df.rename(columns={'distance' : 'travelled_distance'})
    return df.sort_values(['travelled_distance', 'name'] , ascending=[False, True])

